In [4]:
import os, glob, time, requests, io, ssl, certifi, urllib.request
import pandas as pd, numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from math import erf, sqrt

# -------------------------------------------------
# 1) Bring in your ESPN functions from the other notebook
# -------------------------------------------------
%run ESPNDATA.ipynb   # must define: pull_players_and_teams_for_date(yyyymmdd)

# -------------------------------------------------
# 2) Utilities
# -------------------------------------------------
def payoff_from_american(odds):
    o = int(odds)
    return o/100.0 if o>0 else 100.0/abs(o)

def normal_cdf(x):  # Φ(x)
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def p_over_from_mu_sigma(line, mu, sigma):
    if sigma <= 1e-6:
        return float(mu > line)
    z = (line - mu) / sigma
    return 1 - normal_cdf(z)

# -------------------------------------------------
# 3) Ensure team points exist in teams_df (via ESPN summary header)
# -------------------------------------------------
SITE_SUMMARY = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary"

def _j(url):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json()

def _scores_for_event(event_id: str):
    h = _j(f"{SITE_SUMMARY}?event={event_id}").get("header", {})
    comp = (h.get("competitions") or [{}])[0]
    out = []
    for c in comp.get("competitors", []):
        team_name = c.get("team", {}).get("displayName")
        pts = c.get("score")
        if pts is None:
            ls = c.get("linescores") or []
            if ls:
                pts = sum(int(x.get("value") or 0) for x in ls)
        if team_name is not None and pts is not None:
            out.append({"event_id": str(event_id), "team": team_name, "points": int(pts)})
    return out

def ensure_points(teams_df: pd.DataFrame) -> pd.DataFrame:
    df = teams_df.copy()
    for candidate in ("points","pts","score"):
        if candidate in df.columns:
            return df.rename(columns={candidate: "points"}) if candidate != "points" else df

    score_rows = []
    for eid in df["event_id"].astype(str).unique():
        try:
            score_rows.extend(_scores_for_event(eid))
        except Exception:
            continue
        time.sleep(0.06)

    scores = pd.DataFrame(score_rows).dropna(subset=["points"])
    if scores.empty:
        raise ValueError("Couldn't derive team points; inspect teams_df and a sample summary JSON.")

    out = df.merge(scores, on=["event_id","team"], how="left")

    # fallback: lenient team-name match if needed
    if out["points"].isna().any():
        df2 = out[out["points"].isna()].copy()
        ok  = out[out["points"].notna()]
        if not df2.empty:
            df2["team_key"] = df2["team"].str.strip().str.lower()
            scores["team_key"] = scores["team"].str.strip().str.lower()
            df2 = df2.drop(columns=["points"]).merge(
                scores.drop(columns=["team"]).drop_duplicates(["event_id","team_key"]),
                on=["event_id","team_key"], how="left"
            ).drop(columns=["team_key"])
            out = pd.concat([ok, df2], ignore_index=True)
    return out

# -------------------------------------------------
# 4) Build training set from ESPN team totals
# -------------------------------------------------
def pull_team_totals_for_dates(date_list):
    rows = []
    for ds in date_list:
        _, teams_df = pull_players_and_teams_for_date(ds)  # from ESPNDATA.ipynb
        if teams_df.empty:
            continue
        rows.append(teams_df.assign(asof_date=ds))
        time.sleep(0.08)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def build_game_table_from_teams(teams_df):
    t = ensure_points(teams_df.copy())   # ensure 'points'
    t['team_key'] = t['team'].str.strip().str.lower()

    g = t[['event_id','team','team_key','points','asof_date']]
    g_sorted = g.sort_values(['event_id','team_key'])
    pairs = []
    for eid, grp in g_sorted.groupby('event_id'):
        if len(grp) != 2:
            continue
        a, b = grp.iloc[0], grp.iloc[1]
        pairs.append({
            'event_id': eid,
            'team_a': a.team, 'team_b': b.team,
            'team_a_key': a.team_key, 'team_b_key': b.team_key,
            'pts_a': pd.to_numeric(a.points, errors='coerce'),
            'pts_b': pd.to_numeric(b.points, errors='coerce'),
            'asof_date': a.asof_date
        })
    games = pd.DataFrame(pairs).dropna(subset=['pts_a','pts_b'])
    games['total_points'] = games['pts_a'] + games['pts_b']
    return games

def rolling_team_features(games, window=3):
    a = games[['event_id','asof_date','team_a_key','pts_a','pts_b']].rename(
        columns={'team_a_key':'team_key','pts_a':'pts_for','pts_b':'pts_against'})
    b = games[['event_id','asof_date','team_b_key','pts_b','pts_a']].rename(
        columns={'team_b_key':'team_key','pts_b':'pts_for','pts_a':'pts_against'})
    long = pd.concat([a,b], ignore_index=True).sort_values(['team_key','asof_date','event_id'])

    feats = []
    for team, grp in long.groupby('team_key'):
        grp = grp.copy()
        grp['pf_l3'] = grp['pts_for'].shift(1).rolling(window).mean()
        grp['pa_l3'] = grp['pts_against'].shift(1).rolling(window).mean()
        grp['pf_l5'] = grp['pts_for'].shift(1).rolling(5).mean()
        grp['pa_l5'] = grp['pts_against'].shift(1).rolling(5).mean()
        feats.append(grp.assign(team_key=team))
    f = pd.concat(feats, ignore_index=True)

    fa = f[['event_id','team_key','pf_l3','pa_l3','pf_l5','pa_l5']]
    fb = fa.copy()
    games2 = games.merge(fa, left_on=['event_id','team_a_key'], right_on=['event_id','team_key'], how='left') \
                  .drop(columns=['team_key']) \
                  .rename(columns={'pf_l3':'a_pf_l3','pa_l3':'a_pa_l3','pf_l5':'a_pf_l5','pa_l5':'a_pa_l5'})
    games2 = games2.merge(fb, left_on=['event_id','team_b_key'], right_on=['event_id','team_key'], how='left') \
                   .drop(columns=['team_key']) \
                   .rename(columns={'pf_l3':'b_pf_l3','pa_l3':'b_pa_l3','pf_l5':'b_pf_l5','pa_l5':'b_pa_l5'})
    return games2

def train_total_model(train_games):
    features = ['a_pf_l3','a_pa_l3','a_pf_l5','a_pa_l5','b_pf_l3','b_pa_l3','b_pf_l5','b_pa_l5']
    X = train_games[features].fillna(train_games[features].mean())
    y = train_games['total_points']
    model = Ridge(alpha=3.0).fit(X, y)

    # estimate residual sigma with 5-fold CV
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    preds, ys = [], []
    for tr, te in kf.split(X):
        m = Ridge(alpha=3.0).fit(X.iloc[tr], y.iloc[tr])
        p = m.predict(X.iloc[te])
        preds.append(p); ys.append(y.iloc[te].values)
    resid = np.concatenate(ys) - np.concatenate(preds)
    sigma = np.std(resid, ddof=1)
    return model, sigma, features

def build_training_from_dates(n_days=200):
    today = datetime.utcnow().date()
    dates = [(today - timedelta(days=i)).strftime("%Y%m%d") for i in range(n_days)]
    ttot = pull_team_totals_for_dates(dates)
    games = build_game_table_from_teams(ttot)
    games = rolling_team_features(games)
    games = games.dropna(subset=['a_pf_l3','b_pf_l3'])
    return games

# -------------------------------------------------
# 5) READ GAME TOTALS (Over/Under) FROM GOOGLE SHEETS
# -------------------------------------------------
SHEET_CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQtfhqFKMwFDldCgWJp4Lb5wqm71F2EXUdwYD_75VxAMPlyUsoMaWct5KrYwXJyPScMxTKLjonLrEbB/pub?gid=0&single=true&output=csv"

def _pick_col(df, candidates):
    cmap = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns: return c
        if c.lower() in cmap: return cmap[c.lower()]
    return None

def read_totals_from_google_sheet(csv_url: str) -> pd.DataFrame:
    """
    Return one row per game+book with:
      book, home_team_api, away_team_api, point, commence_time,
      price_over, price_under, home_key, away_key
    Supports 'wide' (price_over/price_under) or 'long' (Over/Under rows) layouts.
    """
    # ---- SSL-safe CSV fetch (patch) ----
    try:
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(csv_url, context=ctx, timeout=30) as resp:
            raw = pd.read_csv(io.BytesIO(resp.read()))
    except Exception:
        r = requests.get(csv_url, timeout=30)
        r.raise_for_status()
        raw = pd.read_csv(io.BytesIO(r.content))
    # ------------------------------------

    home_col = _pick_col(raw, ["home_team_api","home_team","home"])
    away_col = _pick_col(raw, ["away_team_api","away_team","away"])
    book_col = _pick_col(raw, ["book","bookmaker"])
    point_col = _pick_col(raw, ["point","total","line","total_points"])
    time_col = _pick_col(raw, ["commence_time","start_time","kickoff"])
    market_col = _pick_col(raw, ["market"])
    name_col  = _pick_col(raw, ["name","label"])          # 'Over'/'Under' in long format
    price_col = _pick_col(raw, ["price","odds","american_odds"])
    over_col  = _pick_col(raw, ["price_over","over_price","overodds","over"])
    under_col = _pick_col(raw, ["price_under","under_price","underodds","under"])

    if home_col is None or away_col is None or book_col is None:
        raise ValueError(f"Missing core columns (home/away/book). Found: {list(raw.columns)}")

    df = raw.copy()
    # If a 'market' column exists, filter to totals
    if market_col and market_col in df.columns:
        mask_tot = df[market_col].astype(str).str.lower().eq("totals")
        if mask_tot.any():
            df = df[mask_tot].copy()

    # Wide format (already has both prices)
    if over_col and under_col and point_col:
        use_cols = [book_col, home_col, away_col, point_col, over_col, under_col] + ([time_col] if time_col else [])
        w = df[use_cols].rename(columns={
            book_col: "book",
            home_col: "home_team_api",
            away_col: "away_team_api",
            point_col: "point",
            (time_col or "commence_time"): "commence_time",
            over_col: "price_over",
            under_col: "price_under",
        })
    else:
        # Long format: join Over/Under rows
        if not (name_col and price_col and point_col):
            raise ValueError(
                "Sheet isn’t a totals layout. Create a totals tab with a 'point' (total line) "
                "and Over/Under prices (either wide or long)."
            )
        on_cols = [book_col, home_col, away_col, point_col] + ([time_col] if time_col else [])
        over  = df[df[name_col].astype(str).str.lower().eq("over") ][on_cols + [price_col]].copy()
        under = df[df[name_col].astype(str).str.lower().eq("under")][on_cols + [price_col]].copy()
        m = over.merge(under, on=on_cols, suffixes=("_over","_under")).rename(columns={
            book_col: "book",
            home_col: "home_team_api",
            away_col: "away_team_api",
            point_col: "point",
            (time_col or "commence_time"): "commence_time",
            f"{price_col}_over":  "price_over",
            f"{price_col}_under": "price_under",
        })
        w = m

    # Clean types & keys
    w["price_over"]  = pd.to_numeric(w["price_over"], errors="coerce").astype("Int64")
    w["price_under"] = pd.to_numeric(w["price_under"], errors="coerce").astype("Int64")
    w["point"] = pd.to_numeric(w["point"], errors="coerce")
    if "commence_time" not in w.columns:
        w["commence_time"] = pd.NaT

    norm = lambda s: str(s).strip().lower()
    w["home_key"] = w["home_team_api"].map(norm)
    w["away_key"] = w["away_team_api"].map(norm)
    return w

# =================================================
# ===== 1) TRAIN
# =================================================
train_games = build_training_from_dates(n_days=200)
model, sigma, FEATURES = train_total_model(train_games)
print("Training games:", train_games.shape, "| Estimated sigma:", round(float(sigma),2))

# =================================================
# ===== 2) READ ODDS FROM GOOGLE SHEETS
# =================================================
odds = read_totals_from_google_sheet(SHEET_CSV_URL)
if odds.empty:
    raise RuntimeError("Odds sheet returned 0 rows. Double-check the tab has totals lines and prices.")
print("Loaded totals from Google Sheets:", len(odds))
print(odds.head(5).to_string(index=False))

# =================================================
# ===== 3) SCORE VS ODDS
# =================================================
latest = train_games.copy()
norm = lambda s: str(s).strip().lower()

def get_latest_feats(team_key, side_prefix='a'):
    cols = [f'{side_prefix}_pf_l3', f'{side_prefix}_pa_l3', f'{side_prefix}_pf_l5', f'{side_prefix}_pa_l5']
    a = latest[latest['team_a_key']==team_key][['event_id','team_a_key']+cols].tail(1)
    if not a.empty:
        return a.iloc[0][cols].values
    cols_b = ['b_pf_l3','b_pa_l3','b_pf_l5','b_pa_l5']
    b = latest[latest['team_b_key']==team_key][['event_id','team_b_key']+cols_b].tail(1)
    if not b.empty:
        return b.iloc[0][cols_b].values
    return [np.nan, np.nan, np.nan, np.nan]

pred_rows = []
for _, r in odds.iterrows():
    home_k = norm(r['home_team_api']); away_k = norm(r['away_team_api'])
    a_feats = get_latest_feats(home_k, 'a')
    b_feats = get_latest_feats(away_k, 'b')
    pred_rows.append({
        'home_key': home_k, 'away_key': away_k,
        'book': r['book'], 'total_line': r['point'],
        'price_over': r['price_over'], 'price_under': r['price_under'],
        'commence_time': r['commence_time'],
        'a_pf_l3': a_feats[0], 'a_pa_l3': a_feats[1], 'a_pf_l5': a_feats[2], 'a_pa_l5': a_feats[3],
        'b_pf_l3': b_feats[0], 'b_pa_l3': b_feats[1], 'b_pf_l5': b_feats[2], 'b_pa_l5': b_feats[3],
    })

pred_df = pd.DataFrame(pred_rows)
X_new = pred_df[FEATURES].fillna(train_games[FEATURES].mean())
pred_df['pred_total_mu'] = model.predict(X_new)
pred_df['pred_sigma'] = sigma

pred_df['p_over']  = pred_df.apply(lambda r: p_over_from_mu_sigma(r['total_line'], r['pred_total_mu'], r['pred_sigma']), axis=1)
pred_df['p_under'] = 1 - pred_df['p_over']

pred_df['payoff_over']  = pred_df['price_over'].map(payoff_from_american)
pred_df['payoff_under'] = pred_df['price_under'].map(payoff_from_american)

pred_df['ev_over']  = pred_df['p_over']  * pred_df['payoff_over']  - (1 - pred_df['p_over'])
pred_df['ev_under'] = pred_df['p_under'] * pred_df['payoff_under'] - (1 - pred_df['p_under'])

pred_df['side']   = np.where(pred_df['ev_over'] >= pred_df['ev_under'], 'Over', 'Under')
pred_df['edge']   = pred_df[['ev_over','ev_under']].max(axis=1)
pred_df['confidence'] = np.where(pred_df['side']=='Over', pred_df['p_over'], pred_df['p_under'])

def kelly_fraction(p, odds_american, q=0.25):
    b = payoff_from_american(int(odds_american))
    f = (p*(b+1)-1)/b
    return max(0.0, q*float(f))

pred_df['kelly_frac'] = np.where(
    pred_df['side']=='Over',
    pred_df.apply(lambda r: kelly_fraction(r['p_over'],  r['price_over']),  axis=1),
    pred_df.apply(lambda r: kelly_fraction(r['p_under'], r['price_under']), axis=1)
)

cols = ['commence_time','book','home_key','away_key','total_line',
        'pred_total_mu','pred_sigma','side','confidence','edge','kelly_frac',
        'price_over','price_under','p_over','p_under']
recos = pred_df[cols].sort_values('edge', ascending=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
print(recos.head(20).to_string(index=False))

# Optional: save recommendations for your app
recos.to_csv("ou_recommendations_upcoming.csv", index=False)


Found 16 games via: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates=2024&seasontype=2&week=1
Rows: 1228
 season  week             team stat_type           player athlete_id  seasontype  event_id C/ATT YDS  AVG  TD INT SACKS
   2024     1 Baltimore Ravens   passing    Lamar Jackson    3916387           2 401671789 26/41 273  6.7   1   0   1-6
   2024     1 Baltimore Ravens   rushing    Lamar Jackson    3916387           2 401671789   NaN 122  7.6   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing    Derrick Henry    3043078           2 401671789   NaN  46  3.5   1 NaN   NaN
   2024     1 Baltimore Ravens   rushing      Zay Flowers    4429615           2 401671789   NaN  14  7.0   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing     Justice Hill    4038441           2 401671789   NaN   3  3.0   0 NaN   NaN
   2024     1 Baltimore Ravens receiving    Isaiah Likely    4361050           2 401671789   NaN 111 12.3   1 NaN   NaN
   2024     1 Baltimore Rave

/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_6285/4067913555.py:169: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()



Players rows: 523
 event_id           team stat_type                player athlete_id C/ATT YDS  AVG  TD INT SACKS   RTG CAR
401772760 Miami Dolphins   passing        Tua Tagovailoa    4241479 20/26 205  7.9   4   0   1-8 138.6 NaN
401772760 Miami Dolphins   rushing         De'Von Achane    4429160   NaN  67  3.7   0 NaN   NaN   NaN  18
401772760 Miami Dolphins   rushing       Ollie Gordon II    4711533   NaN  46  4.6   0 NaN   NaN   NaN  10
401772760 Miami Dolphins   rushing         Jaylen Wright    4682745   NaN   7  2.3   0 NaN   NaN   NaN   3
401772760 Miami Dolphins receiving         Jaylen Waddle    4372016   NaN  99 19.8   1 NaN   NaN   NaN NaN
401772760 Miami Dolphins receiving      Malik Washington    4569603   NaN  36  9.0   1 NaN   NaN   NaN NaN
401772760 Miami Dolphins receiving         De'Von Achane    4429160   NaN  24  4.8   1 NaN   NaN   NaN NaN
401772760 Miami Dolphins receiving       Ollie Gordon II    4711533   NaN  20 20.0   1 NaN   NaN   NaN NaN
401772760 Miami Do

TypeError: cannot safely cast non-equivalent float64 to int64